# Archivus – Parallel Upload Load Test

Pushes **up to 20 files at once** at the upload endpoints and reports what breaks
first. Both upload paths are covered, because they fail in different ways:

| Path | Endpoint | Where the pressure lands |
|------|----------|--------------------------|
| Plain | `/storage/file/upload` | request goroutines, multipart spooling, DB writes |
| Chunked | `/storage/file/upload/chunk/*` | chunk staging disk, assembly, the pending-upload queue |

**What to watch for**

- `503` — deliberate backpressure: the staging backlog hit its limit. This is the
  server behaving correctly, not a bug.
- `database is locked` in a 400/500 body — SQLite write contention.
- Latency that grows linearly with concurrency — requests are queueing, not running
  in parallel.
- Files stuck in `pending`/`uploading` long after the uploads return — the
  background drain is falling behind ingest.

**Pre-requisites**

- Server running on `http://localhost:8080`
- Free disk for the generated corpus (see the size knobs in section 2)

In [ ]:
import requests, json, os, time, threading, statistics
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

BASE = 'http://localhost:8080'

def show(resp):
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    print(f'Status : {resp.status_code}')
    print(f'Body   : {json.dumps(body, indent=2)}')
    return body

token    = None
drive_id = None

def auth():
    return {'Authorization': f'Bearer {token}'}

# One Session per worker thread. requests.Session is not thread-safe, and the
# default urllib3 pool caps at 10 connections per host — sharing one Session
# across 20 threads would quietly serialize half the run and make the numbers
# meaningless.
_local = threading.local()

def session():
    s = getattr(_local, 's', None)
    if s is None:
        s = requests.Session()
        s.mount('http://', requests.adapters.HTTPAdapter(pool_maxsize=32))
        _local.s = s
    return s

def run_parallel(fn, items, workers):
    'Run fn over items with `workers` threads; return (results, wall_seconds).'
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        results = list(ex.map(fn, items))
    return results, time.perf_counter() - t0

def summarize(label, results, total_bytes, wall):
    lat = sorted(r['secs'] for r in results)
    def pct(p):
        return lat[min(len(lat) - 1, int(len(lat) * p))] if lat else 0.0
    ok  = [r for r in results if r['status'] == 200]
    bad = [r for r in results if r['status'] != 200]
    print(f'{label}')
    print(f'  files      : {len(results)}   ok={len(ok)}   failed={len(bad)}')
    print(f'  wall       : {wall:.2f}s')
    print(f'  throughput : {total_bytes / wall / 1e6:.1f} MB/s')
    print(f'  latency    : p50={pct(.50):.2f}s  p95={pct(.95):.2f}s  max={lat[-1] if lat else 0:.2f}s')
    print(f'  statuses   : {dict(Counter(r["status"] for r in results))}')
    for msg, n in Counter((r.get('error') or '')[:120] for r in bad).items():
        if msg:
            print(f'    {n}x {msg}')
    return {'label': label, 'n': len(results), 'ok': len(ok), 'failed': len(bad),
            'wall': wall, 'mbps': total_bytes / wall / 1e6,
            'p50': pct(.50), 'p95': pct(.95), 'max': lat[-1] if lat else 0.0,
            'codes': dict(Counter(r['status'] for r in results))}

runs = []   # summary rows, printed as a table at the end

## 1 · Auth setup

Register + login as an admin user to get a token and the owner drive ID.
A duplicate-register 400 is harmless if the user already exists.

In [ ]:
requests.post(f'{BASE}/auth/register', json={
    'username': 'samar', 'password': 'password12', 'pin': '123456',
    'email': 'samar@example.com', 'user_type': 'business', 'is_admin': True,
})

resp = requests.post(f'{BASE}/auth/login', json={'username': 'samar', 'pin': '123456'})
token = resp.json()['token']

resp = requests.get(f'{BASE}/auth/user/info', headers=auth())
drives = resp.json().get('drives', [])
drive_id = drives[0]['DriveID'] if drives else None
print(f'drive_id : {drive_id}')
assert drive_id, 'drive_id is None - register/login failed'

## 2 · Build the corpus

Synthetic files so the test is self-contained and repeatable. Sizes are the knobs:
turn them up to make the run bandwidth-bound rather than round-trip-bound.

Defaults write **~360 MB** into `tmp/parallel/`. `PLAIN_MB` stays small because
that path holds the whole request in flight; `CHUNK_MB` is set above the sync
client's 64 MB chunking threshold so these are the same shape as real large uploads.

In [ ]:
N_FILES   = 20          # peak concurrency the notebook drives
PLAIN_MB  = 2           # per-file size for the plain upload tests
CHUNK_MB  = 16          # per-file size for the chunked upload test
CHUNK_SIZE = 8 << 20    # 8 MB, matching the sync client's default

CORPUS = 'tmp/parallel'
os.makedirs(CORPUS, exist_ok=True)

# Repeat one random megabyte rather than generating hundreds of them: os.urandom
# is slow enough at this size to dominate the cell, and the bytes only need to
# be transported, not compressed or deduplicated.
_block = os.urandom(1 << 20)

def make_file(path, size_mb):
    if os.path.exists(path) and os.path.getsize(path) == size_mb << 20:
        return path
    with open(path, 'wb') as f:
        for _ in range(size_mb):
            f.write(_block)
    return path

plain_files = [make_file(f'{CORPUS}/plain_{i:02d}.bin', PLAIN_MB) for i in range(N_FILES)]
chunk_files = [make_file(f'{CORPUS}/chunk_{i:02d}.bin', CHUNK_MB) for i in range(N_FILES)]

PLAIN_BYTES = PLAIN_MB << 20
CHUNK_BYTES = CHUNK_MB << 20
print(f'{len(plain_files)} x {PLAIN_MB} MB plain   = {len(plain_files)*PLAIN_MB} MB')
print(f'{len(chunk_files)} x {CHUNK_MB} MB chunked = {len(chunk_files)*CHUNK_MB} MB')
print(f'corpus on disk: {sum(os.path.getsize(p) for p in plain_files + chunk_files) / 1e6:.0f} MB')

In [ ]:
# Destination folders, scoped to this run. The chunked path rejects an unknown
# folder at init, so they have to exist before any upload runs.
#
# The run-unique prefix is not cosmetic: deleting a folder removes its objects
# and its directory row but leaves the FileMetadata rows for the files inside it,
# so re-uploading to a path a previous run cleaned up fails with "archive
# previous version ... CopyObject" — the DB thinks the old object is still there.
# A fresh prefix per run keeps that from being mistaken for a concurrency bug.
RUN_ID       = time.strftime('%H%M%S')
PLAIN_DIR    = f'loadtest/{RUN_ID}/plain'
CHUNKED_DIR  = f'loadtest/{RUN_ID}/chunked'

for folder in (PLAIN_DIR, CHUNKED_DIR):
    r = requests.post(f'{BASE}/storage/folder/create', headers=auth(),
                      json={'path': folder, 'driveId': drive_id})
    print(f'{folder:28s} -> {r.status_code}')
    assert r.status_code == 200, r.text

## 3 · Baseline: one at a time

A reference number for the sections below. If 20-way concurrency does not beat
this, nothing is actually running in parallel.

In [ ]:
def upload_plain(path, folder=None):
    folder = folder or PLAIN_DIR
    name = os.path.basename(path)
    t0 = time.perf_counter()
    try:
        with open(path, 'rb') as fh:
            r = session().post(f'{BASE}/storage/file/upload', headers=auth(),
                               data={'folderPath': folder, 'driveId': drive_id},
                               files=[('files', (name, fh, 'application/octet-stream'))],
                               timeout=600)
        return {'file': name, 'status': r.status_code, 'secs': time.perf_counter() - t0,
                'error': None if r.status_code == 200 else r.text[:200]}
    except Exception as e:
        return {'file': name, 'status': -1, 'secs': time.perf_counter() - t0,
                'error': f'{type(e).__name__}: {e}'}

sample = plain_files[:5]
t0 = time.perf_counter()
results = [upload_plain(p) for p in sample]
runs.append(summarize('sequential x5 (plain)', results, len(sample) * PLAIN_BYTES,
                      time.perf_counter() - t0))

## 4 · Ramp concurrency on the plain upload path

1 → 5 → 10 → 20 simultaneous uploads. Each level re-uploads the same 20 files, so
every level moves identical bytes and the numbers are directly comparable.

Re-uploading the same names also exercises the overwrite/versioning path, which
is the write-heaviest thing the DB does — a good way to shake out lock contention.

In [ ]:
for workers in (1, 5, 10, 20):
    results, wall = run_parallel(upload_plain, plain_files, workers)
    runs.append(summarize(f'{workers:>2} concurrent (plain)', results,
                          len(plain_files) * PLAIN_BYTES, wall))
    print()

## 5 · Twenty concurrent chunked uploads

The path that actually stresses the server's disk: each file is staged as chunks,
then assembled into a *second* full copy, then queued for the storage backend.
Twenty at once means up to 20 × 2 × `CHUNK_MB` of staging in flight.

Each thread drives one file through `init` → `part` × N → `complete`, exactly as a
real client would.

In [ ]:
def upload_chunked(path, folder=None, chunk_size=CHUNK_SIZE):
    folder = folder or CHUNKED_DIR
    name = os.path.basename(path)
    size = os.path.getsize(path)
    total = max(1, (size + chunk_size - 1) // chunk_size)
    t0 = time.perf_counter()
    s = session()

    def fail(stage, r):
        return {'file': name, 'stage': stage, 'status': r.status_code,
                'secs': time.perf_counter() - t0, 'error': r.text[:200], 'uploadId': None}
    try:
        r = s.post(f'{BASE}/storage/file/upload/chunk/init', headers=auth(), timeout=60,
                   json={'filename': name, 'driveId': drive_id, 'folderPath': folder,
                         'contentType': 'application/octet-stream',
                         'size': size, 'totalChunks': total})
        if r.status_code != 200:
            return fail('init', r)
        upload_id = r.json()['uploadId']

        with open(path, 'rb') as fh:
            for i in range(total):
                fh.seek(i * chunk_size)
                blob = fh.read(chunk_size)
                r = s.post(f'{BASE}/storage/file/upload/chunk/part', headers=auth(), timeout=300,
                           data={'uploadId': upload_id, 'chunkIndex': str(i)},
                           files=[('chunk', (f'chunk-{i}', blob))])
                if r.status_code != 200:
                    return fail(f'part[{i}/{total}]', r)

        r = s.post(f'{BASE}/storage/file/upload/chunk/complete', headers=auth(),
                   json={'uploadId': upload_id}, timeout=600)
        if r.status_code != 200:
            return fail('complete', r)
        return {'file': name, 'stage': 'complete', 'status': 200,
                'secs': time.perf_counter() - t0, 'error': None, 'uploadId': upload_id,
                'fileStatus': r.json().get('status')}
    except Exception as e:
        return {'file': name, 'stage': 'exception', 'status': -1,
                'secs': time.perf_counter() - t0, 'error': f'{type(e).__name__}: {e}',
                'uploadId': None}

In [ ]:
chunk_results, wall = run_parallel(upload_chunked, chunk_files, N_FILES)
runs.append(summarize(f'{N_FILES} concurrent (chunked, accept)', chunk_results,
                      len(chunk_files) * CHUNK_BYTES, wall))
chunk_accept_wall = wall

# That MB/s figure is the *acceptance* rate, not a storage rate. On an
# object-storage backend /complete returns as soon as the bytes are assembled on
# local disk, so this measures how fast the server can be handed work — section 6
# measures how fast it actually gets rid of it.
print('\nNOTE: throughput above is ingest to local staging, not bytes persisted.')

# Where the failures landed matters more than the count: init means it was
# refused up front, complete means the bytes were already spent.
print('\nby stage :', dict(Counter(r.get('stage') for r in chunk_results)))
print('accepted :', dict(Counter(r.get('fileStatus') for r in chunk_results if r['status'] == 200)))
for r in chunk_results:
    if r['status'] != 200:
        print(f'  FAIL {r["file"]:16s} {r["stage"]:14s} {r["status"]}  {(r["error"] or "")[:120]}')

## 6 · Watch the queue drain

`/complete` returning 200 means *accepted*, not *stored*. On an object-storage
backend the bytes are still on local disk waiting for a background worker, and the
listing reports that as `pending` / `uploading`.

This poll is where ingest-versus-drain shows up. If the counts sit still, the
workers are not keeping up.

In [ ]:
def status_counts(folder=None):
    folder = folder or CHUNKED_DIR
    r = requests.post(f'{BASE}/storage/files', headers=auth(),
                      json={'path': folder, 'driveId': drive_id, 'pageSize': 500})
    if r.status_code != 200:
        return Counter({f'<list failed {r.status_code}>': 1})
    files = [e for e in r.json().get('files', []) if not e.get('IsDir')]
    return Counter(e.get('UploadStatus') or 'ready' for e in files)

DEADLINE = 300   # seconds to wait for the queue to clear
t0 = time.perf_counter()
last = None
while True:
    counts = status_counts()
    elapsed = time.perf_counter() - t0
    if counts != last:
        print(f'{elapsed:6.1f}s  {dict(counts)}')
        last = counts
    if not (counts['pending'] or counts['uploading']):
        print(f'\nqueue clear after {elapsed:.1f}s')
        break
    if elapsed > DEADLINE:
        print(f'\nSTILL DRAINING after {DEADLINE}s - ingest is outrunning the workers')
        break
    time.sleep(2)

print('final :', dict(status_counts()))

# The number that actually matters: bytes persisted per second, end to end.
# Anything the accept phase "achieved" beyond this was just buffered on disk.
total_bytes = len(chunk_files) * CHUNK_BYTES
end_to_end = chunk_accept_wall + elapsed
print(f'\naccept   : {total_bytes / chunk_accept_wall / 1e6:6.1f} MB/s  (into staging)')
print(f'drain    : {total_bytes / elapsed / 1e6:6.1f} MB/s  (staging -> storage)')
print(f'end-to-end: {total_bytes / end_to_end / 1e6:5.1f} MB/s  over {end_to_end:.1f}s')
print('\nThe drain rate is the real ceiling. If accept is far above it, clients are')
print('being told "done" for work the server has only queued - which is exactly')
print('what the staging backlog limit exists to bound.')
runs.append({'label': f'{N_FILES} concurrent (chunked, drain)', 'n': len(chunk_files),
             'ok': len(chunk_files), 'failed': 0, 'wall': elapsed,
             'mbps': total_bytes / elapsed / 1e6, 'p50': 0.0, 'p95': 0.0, 'max': 0.0,
             'codes': {}})

## 7 · Backpressure and failures

`503` is the server refusing new work because the staging backlog is full. Seeing
it is not a failure of the test — it is the mechanism working, and it means the
client should retry rather than give up. The chunked flow is resumable, so a
refused `complete` can simply be retried later with no bytes re-sent.

Anything *else* non-200 is worth reading closely.

In [ ]:
throttled = [r for r in chunk_results if r['status'] == 503]
print(f'503 backpressure : {len(throttled)}')
for r in throttled[:5]:
    print(f'   {r["file"]:16s} at {r["stage"]}')

other = [r for r in chunk_results if r['status'] not in (200, 503)]
print(f'\nother failures   : {len(other)}')
for r in other[:10]:
    print(f'   {r["file"]:16s} {r["stage"]:14s} {r["status"]}  {(r["error"] or "")[:160]}')

locked = [r for r in chunk_results if 'database is locked' in (r.get('error') or '')]
print(f'\nSQLite lock errors : {len(locked)}   <- should be 0; if not, WAL/busy_timeout is not in effect')

## 8 · Summary

In [ ]:
print(f'{"run":28s} {"n":>3} {"ok":>3} {"fail":>4} {"wall":>7} {"MB/s":>7} {"p50":>7} {"p95":>7} {"max":>7}')
print('-' * 84)
for r in runs:
    print(f'{r["label"]:28s} {r["n"]:3d} {r["ok"]:3d} {r["failed"]:4d} '
          f'{r["wall"]:6.2f}s {r["mbps"]:7.1f} {r["p50"]:6.2f}s {r["p95"]:6.2f}s {r["max"]:6.2f}s')

print('\nRead it like this:')
print('  MB/s flat as workers rise      -> saturated somewhere (uplink, disk, or a serialized stage)')
print('  p95 rising in step with workers -> requests are queueing, not running concurrently')
print('  failures only at 20            -> that is your ceiling; find it before production does')

## 9 · Cleanup

Removes both the uploaded folders and the generated corpus. Skip it if you want to
inspect the server state after a run.

In [ ]:
for folder in (CHUNKED_DIR, PLAIN_DIR, f'loadtest/{RUN_ID}'):
    r = requests.post(f'{BASE}/storage/folder/delete', headers=auth(),
                      json={'path': folder, 'driveId': drive_id})
    print(f'{folder:28s} -> {r.status_code}')

import shutil
shutil.rmtree(CORPUS, ignore_errors=True)
print(f'removed {CORPUS}')